# AD mouse brain

This notebook shows the complete `admouse` analysis: data preparation,
model training, downstream analysis, and the commands used for the paper
figures. Edit the paths in **Setup** before starting a run.

## Setup

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from CytoBridge.workflow import (
    WorkflowOptions,
    build_workflow_plan,
    load_workflow_config,
    render_workflow_plan,
    run_workflow,
)
DATASET_CONFIG = 'admouse'
RAW_H5AD = Path("data/admouse_raw.h5ad")
OUTPUT_DIR = Path("tutorial_outputs/admouse")
ALIGNED_H5AD = OUTPUT_DIR / "preprocess" / 'admouse_aligned.h5ad'
MODEL_DIR = OUTPUT_DIR / "training"


RUN_PREPARATION = False
RUN_PREPROCESS_AND_TRAIN = False
RUN_DOWNSTREAM = False

In [2]:
config, config_source = load_workflow_config(DATASET_CONFIG)
dataset = config["dataset"]
scientific = config["scientific"]
downstream = config["downstream"]

pd.DataFrame(
    {
        "setting": [
            "dataset",
            "configuration",
            "raw time column",
            "cell annotation",
            "observed training times",
            "classifier neighbors",
        ],
        "value": [
            dataset["display_name"],
            config_source,
            config["preprocess"]["time_key"],
            dataset["annotation_key"],
            ", ".join(map(str, downstream["observed"])),
            scientific["classifier_k"],
        ],
    }
)

,setting,value
0,dataset,AD mouse brain
1,configuration,example configuration: admouse
2,raw time column,Timepoint
3,cell annotation,major_annotation
4,observed training times,"0.0, 1.0, 2.0"
5,classifier neighbors,1


## Data preparation

The dataset configuration records the count layer, time mapping, spatial
coordinates, and alignment settings. The command below reads the raw H5AD and
writes the aligned H5AD and edge model used for training.

### 1. preprocess

```text
cytobridge workflow --config admouse --step preprocess --input-h5ad <raw.h5ad> --output-dir <run>
```

Input: `raw H5AD and the dataset configuration`

Creates: `<run>/preprocess/admouse_aligned.h5ad; <run>/preprocess/edge_classifier/admouse_edge_model.pt; preprocessing records`

Continue with: `training`

In [3]:
preparation_options = WorkflowOptions(
    input_h5ad=RAW_H5AD,
    output_dir=OUTPUT_DIR,
    steps=("preprocess",),
)
preparation_plan = build_workflow_plan(
    config,
    source=config_source,
    options=preparation_options,
)
print(render_workflow_plan(preparation_plan))

CytoBridge workflow plan
dataset: AD mouse brain (admouse)
config: example configuration: admouse
model settings: alpha_spatial=10, alpha_express=0.015, seed=42, classifier_k=1
steps:
  preprocess: ready (GPU recommended for spatial alignment)
    output: tutorial_outputs/admouse/preprocess/admouse_aligned.h5ad
    note: The AD raw-H5AD workflow fits a learned edge predictor from the seven complete ligand-receptor pairs represented by the targeted panel and uses the validation-selected threshold 0.9956824779510498. Broader CCI analyses require a more complete panel. The packaged all-spatial configuration is the corresponding no-LR-prior ablation.
    edge predictor: not requested during preprocessing
  train: skipped; add --train to run (GPU required for training)
  downstream: skipped (GPU recommended)


In [4]:
if RUN_PREPARATION:
    if not RAW_H5AD.is_file():
        raise FileNotFoundError(f"Update RAW_H5AD before preprocessing: {RAW_H5AD}")
    preparation_result = run_workflow(config, options=preparation_options)
    preparation_result
else:
    print("Data preparation is off. Set RUN_PREPARATION = True to run it.")

Data preparation is off. Set RUN_PREPARATION = True to run it.


## Training

The full run starts from the raw H5AD, writes the aligned data, fits the
interaction edge model when needed, and trains CytoBridge. Training requires a
CUDA-capable environment.

### 1. preprocess and train

```text
cytobridge workflow --config admouse --step preprocess --step train --train --input-h5ad <raw.h5ad> --output-dir <run> --device cuda
```

Input: `raw H5AD, dataset configuration, and LR database`

Creates: `<run>/training/<stage>/best_model.pth or score_model.pth; <run>/training/adata.h5ad; training_history.csv; training_run_summary.json`

Continue with: `downstream`

In [5]:
training_options = WorkflowOptions(
    input_h5ad=RAW_H5AD,
    output_dir=OUTPUT_DIR,
    steps=("preprocess", "train"),
    train=True,
)
training_plan = build_workflow_plan(
    config,
    source=config_source,
    options=training_options,
)
print(render_workflow_plan(training_plan))

CytoBridge workflow plan
dataset: AD mouse brain (admouse)
config: example configuration: admouse
model settings: alpha_spatial=10, alpha_express=0.015, seed=42, classifier_k=1
steps:
  preprocess: ready (GPU recommended for spatial alignment)
    output: tutorial_outputs/admouse/preprocess/admouse_aligned.h5ad
    note: The AD raw-H5AD workflow fits a learned edge predictor from the seven complete ligand-receptor pairs represented by the targeted panel and uses the validation-selected threshold 0.9956824779510498. Broader CCI analyses require a more complete panel. The packaged all-spatial configuration is the corresponding no-LR-prior ablation.
    edge predictor: will be trained automatically
      graph database: package: CytoBridge/workflow_databases/CellChatDB.ligrec.mouse.csv
      database source: included CellChatDB resource
      interaction cutoff: 0.012106042891492197
      decision threshold source: validation-selected during de novo training
      output: tutorial_outputs

In [6]:
if RUN_PREPROCESS_AND_TRAIN:
    if not RAW_H5AD.is_file():
        raise FileNotFoundError(f"Update RAW_H5AD before training: {RAW_H5AD}")
    training_result = run_workflow(config, options=training_options)
    training_result
else:
    print("Training is off. Set RUN_PREPROCESS_AND_TRAIN = True to start it.")

Training is off. Set RUN_PREPROCESS_AND_TRAIN = True to start it.


## Downstream analysis

Downstream analysis reads the aligned H5AD and fitted model from the training
directory. It writes generated states, velocity, growth, composition,
communication, ligand–receptor tables, and standard figures.

### 1. downstream

```text
cytobridge workflow --config admouse --step downstream --aligned-h5ad <run>/preprocess/admouse_aligned.h5ad --model-dir <run>/training --output-dir <run>
```

Input: `aligned H5AD; <run>/training; dataset-matched LR database`

Creates: `<run>/downstream/summary.json; slice_data/*.h5ad; velocity/velocity_components.npz; growth/growth_by_cell.csv; composition/celltype_composition.csv; communication and ligand_receptor tables; standard figures`

Continue with: `paper-specific continuation shown in the paper-figure notebook`

In [7]:
downstream_options = WorkflowOptions(
    aligned_h5ad=ALIGNED_H5AD,
    model_dir=MODEL_DIR,
    output_dir=OUTPUT_DIR,
    steps=("downstream",),
)
downstream_plan = build_workflow_plan(
    config,
    source=config_source,
    options=downstream_options,
)
print(render_workflow_plan(downstream_plan))

CytoBridge workflow plan
dataset: AD mouse brain (admouse)
config: example configuration: admouse
model settings: alpha_spatial=10, alpha_express=0.015, seed=42, classifier_k=1
steps:
  preprocess: skipped (GPU for spatial alignment)
  train: skipped; add --train to run (GPU required for training)
  downstream: ready (GPU recommended for SDE simulation and classifier fitting)
    model format: current
    output: tutorial_outputs/admouse/downstream
    generated states: observed times=[0.0, 1.0, 2.0], additional times=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.1, 2.2, 2.3, 2.4, 2.5]
      simulation settings: dt=0.01, sigma=0.03, daughter noise=0, growth alpha=1
    interpolation and classification: enabled
    time-slice velocity: enabled
    growth: enabled when present in the model
    cell-type composition: enabled
    sparse communication: enabled
    standard figures: enabled
      note: snapshots, mosaic, growth, composition, and

In [8]:
if RUN_DOWNSTREAM:
    missing = [path for path in (ALIGNED_H5AD, MODEL_DIR) if not path.exists()]
    if missing:
        raise FileNotFoundError(f"Missing aligned data or model directory: {missing}")
    downstream_result = run_workflow(config, options=downstream_options)
    downstream_result
else:
    print("Downstream analysis is off. Set RUN_DOWNSTREAM = True to run it.")

Downstream analysis is off. Set RUN_DOWNSTREAM = True to run it.


## Paper figures

Continue with these commands to calculate the values used in the paper. Each
step states which downstream files it reads and which paper notebook uses its
output.

- [Interaction-prior ablation](../paper_figures/lr_prior_ablation_stvcr.ipynb)
- [Five-dataset benchmark](../paper_figures/loto_benchmark.ipynb)
- [Training histories](../paper_figures/training_histories.ipynb)

### 1. calculate the available AD figure set

Used for: Main Figure 6; S26-S28

```text
output/admouse_article_figure_replication_20260814/make_admouse_article_figures.py
```

Input: `continuous-t0 gene, LR, attention, perturbation, snapshot, and GO tables`

Creates: `AD article-style vector PDF/PNG pages and derived GO tables`

Continue with: `compare against the manuscript Main Figure 6 and S26-S28 assets`

This builder reproduces the available AD analyses, but the exact S26-S28 page assembly has not been linked to it.

### 2. prepare temporal NicheNet inputs

Used for: S29

```text
python scripts/prepare_temporal_nichenet_inputs.py --help
```

Input: `aligned AD states; interval definitions; NicheNet prior and receiver programs`

Creates: `one NicheNet input directory per interval`

Continue with: `run temporal NicheNet`

### 3. run temporal NicheNet

Used for: S29

```text
Rscript scripts/run_temporal_nichenet_reference.R --help
```

Input: `prepared NicheNet interval directories and the NicheNet prior`

Creates: `interval-level ligand activity and LR-network tables`

Continue with: `compare with CytoBridge`

### 4. compare NicheNet with CytoBridge

Used for: S29

```text
python scripts/compare_cytobridge_to_temporal_nichenet.py --help
```

Input: `NicheNet results and matching CytoBridge LR tables`

Creates: `comparison tables used for plotting`

Continue with: `identify the exact inputs used by the current ad_supp3.pdf`

The NicheNet calculations are available, but the exact inputs used to assemble the current S29 PDF have not been identified.

### 5. Spp1 perturbation analysis and plotting

Used for: S30

```text
output/admouse_article_figure_replication_20260814/make_admouse_article_figures.py and its retained reference notebook
```

Input: `continuous-t0 perturbation results and module-score tables`

Creates: `candidate perturbation tables and article-style figures`

Continue with: `exact manuscript ad_supp4.pdf generator`

The perturbation calculations are available, but the exact inputs and contrast used for the current S30 PDF have not been identified.

## Saved files

- Aligned data: `tutorial_outputs/admouse/preprocess/admouse_aligned.h5ad`
- Training directory: `tutorial_outputs/admouse/training`
- Downstream directory: `tutorial_outputs/admouse/downstream`